# Day4_01C_LangGraph_Conditional_Edges_Quality_Loop

## LangGraph - Conditional Routing and Revision Loops

**Duration:** 25–30 minutes  
**Audience:** Engineering Faculty  
**Environment:** VS Code Notebook  
**Session:** Day 4 - LangGraph Hands-on  

---

## Learning Objectives

By the end of this notebook, participants will be able to:

- Understand why conditional edges are important.
- Build a graph with decision-based routing.
- Create a simple quality gate.
- Implement a revision loop.
- Prevent infinite loops using a revision counter.
- Relate LangGraph loops to academic and enterprise review workflows.

---

## Previous Notebook Recap

In the previous notebook, we built a simple LangGraph workflow:

```text
START
  ↓
Create Lesson Plan
  ↓
Create Summary
  ↓
END
```

That was a straight-line workflow.

Now we will build a smarter workflow.

This workflow can decide:

> Is the output good enough, or should it be revised?


# 1. Real-World Story: Lesson Plan Review

Imagine a faculty member creates a lesson plan.

Before using it in class, the lesson plan should be reviewed.

If the quality is good, it can be used.

If the quality is poor, it should be revised.

```text
Create Lesson Plan
   ↓
Review Lesson Plan
   ↓
Score >= 8?
   ├── Yes → Finalize
   └── No → Revise → Review Again
```

This is exactly the kind of workflow LangGraph is good at.


# 2. What We Will Build

We will build this graph:

```text
START
  ↓
create_draft
  ↓
review_draft
  ↓
quality_decision
   ├── approved → END
   └── revise → revise_draft → review_draft
```

This graph has:

- State
- Nodes
- Edges
- Conditional edge
- Loop
- Exit condition


# 3. Why This Matters

Most real workflows are not simple straight lines.

They include decisions like:

- Is the document approved?
- Is the risk high?
- Is human approval needed?
- Should we retry?
- Is quality good enough?
- Should the output be revised?

Conditional edges allow the graph to choose the next step.


# 4. Setup

Install dependencies if needed:

```bash
pip install langgraph
```

This notebook keeps the logic simple using Python functions.

We are not adding LLM calls yet because the main goal is to understand conditional routing.


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 5. Define the State

Our workflow state will contain:

- topic
- draft
- feedback
- quality_score
- revision_count
- final_status

The revision count is important.

It prevents infinite loops.


In [ ]:
class LessonReviewState(TypedDict):
    topic: str
    draft: str
    feedback: str
    quality_score: int
    revision_count: int
    final_status: str

# 6. Node 1 - Create Draft

This node creates the first draft of the lesson plan.


In [ ]:
def create_draft(state: LessonReviewState):
    topic = state["topic"]

    draft = f"""
    Lesson Draft: {topic}

    This lesson introduces {topic}.
    It includes a basic explanation and one example.

    Sections:
    1. Introduction
    2. Basic explanation
    3. Example
    """

    return {
        "draft": draft,
        "revision_count": 0,
        "final_status": "Draft created"
    }

# 7. Node 2 - Review Draft

This node reviews the draft.

For teaching simplicity, we will use rule-based scoring.

In a real Agentic AI system, this review could be performed by:

- an LLM reviewer
- a human reviewer
- a rubric engine
- a policy checker


In [ ]:
def review_draft(state: LessonReviewState):
    draft = state["draft"].lower()
    revision_count = state["revision_count"]

    score = 5
    feedback_items = []

    if "learning objectives" in draft:
        score += 1
    else:
        feedback_items.append("Add learning objectives.")

    if "activity" in draft:
        score += 1
    else:
        feedback_items.append("Add a classroom activity.")

    if "quiz" in draft:
        score += 1
    else:
        feedback_items.append("Add a short quiz.")

    if "summary" in draft:
        score += 1
    else:
        feedback_items.append("Add a summary.")

    if "enterprise" in draft:
        score += 1
    else:
        feedback_items.append("Add an enterprise example.")

    if score >= 8:
        feedback = "Good quality. Ready for use."
    else:
        feedback = "Needs improvement: " + " ".join(feedback_items)

    return {
        "quality_score": score,
        "feedback": feedback,
        "final_status": f"Reviewed after {revision_count} revision(s)"
    }

# 8. Node 3 - Revise Draft

This node improves the draft based on feedback.

In a real system, this could be an LLM revision Agent.

For teaching, we simulate revision by adding missing sections.


In [ ]:
def revise_draft(state: LessonReviewState):
    draft = state["draft"]
    revision_count = state["revision_count"] + 1

    improved_draft = draft + f"""

    Revision {revision_count}: Improvements Added

    Learning Objectives:
    - Understand the core idea of {state["topic"]}.
    - Identify one classroom use case.
    - Identify one enterprise use case.

    Classroom Activity:
    Ask students to compare a chatbot and an AI agent.

    Short Quiz:
    1. What is the main idea of {state["topic"]}?
    2. Give one example of where it can be used.

    Enterprise Example:
    A customer support agent can use tools to check ticket status and suggest next actions.

    Summary:
    {state["topic"]} helps students understand how AI systems can reason, act, and improve workflows.
    """

    return {
        "draft": improved_draft,
        "revision_count": revision_count,
        "final_status": "Draft revised"
    }

# 9. Conditional Decision Function

This function decides where the graph should go next.

If quality score is high enough, end the graph.

If quality score is low and revision count is below limit, revise.

If too many revisions happened, end anyway.

This protects us from infinite loops.


In [ ]:
def quality_decision(state: LessonReviewState):
    score = state["quality_score"]
    revision_count = state["revision_count"]

    if score >= 8:
        return "approved"

    if revision_count >= 2:
        return "stop_after_max_revisions"

    return "revise" 

# 10. Build the Graph

Now we create the graph and add nodes.


In [ ]:
builder = StateGraph(LessonReviewState)

builder.add_node("create_draft", create_draft)
builder.add_node("review_draft", review_draft)
builder.add_node("revise_draft", revise_draft)

# 11. Add Normal Edges

The graph starts by creating a draft, then reviewing it.


In [ ]:
builder.add_edge(START, "create_draft")
builder.add_edge("create_draft", "review_draft")

# 12. Add Conditional Edge

This is the most important part.

After `review_draft`, the graph should decide:

- approved → END
- revise → revise_draft
- stop_after_max_revisions → END


In [ ]:
builder.add_conditional_edges(
    "review_draft",
    quality_decision,
    {
        "approved": END,
        "revise": "revise_draft",
        "stop_after_max_revisions": END,
    }
)

builder.add_edge("revise_draft", "review_draft")

# 13. Compile the Graph

In [ ]:
lesson_review_graph = builder.compile()

print("Conditional graph compiled successfully.")

# 14. Run the Graph

The first draft is intentionally incomplete.

The graph should review it, revise it, and review again.


In [ ]:
initial_state = {
    "topic": "Agentic AI",
    "draft": "",
    "feedback": "",
    "quality_score": 0,
    "revision_count": 0,
    "final_status": ""
}

final_state = lesson_review_graph.invoke(initial_state)

final_state

# 15. Display the Final Output

In [ ]:
print("=" * 80)
print("LANGGRAPH QUALITY LOOP OUTPUT")
print("=" * 80)

print("\nTOPIC")
print("-" * 80)
print(final_state["topic"])

print("\nQUALITY SCORE")
print("-" * 80)
print(final_state["quality_score"])

print("\nREVISION COUNT")
print("-" * 80)
print(final_state["revision_count"])

print("\nFEEDBACK")
print("-" * 80)
print(final_state["feedback"])

print("\nFINAL STATUS")
print("-" * 80)
print(final_state["final_status"])

print("\nFINAL DRAFT")
print("-" * 80)
print(final_state["draft"])

# 16. What Did We Just Build?

We built a graph that can make decisions.

```text
Draft
  ↓
Review
  ↓
Quality Decision
  ├── Good enough → End
  └── Not good enough → Revise → Review again
```

This is much closer to real workflows than a straight-line chain.


# 17. Why the Revision Counter Matters

Without a revision counter, a workflow could loop forever.

Example:

```text
Review → Revise → Review → Revise → Review → ...
```

So we added:

```python
revision_count
```

and stopped after 2 revisions.

This is an important enterprise design principle.


# 18. Modify the Quality Threshold

Currently we approve when score >= 8.

Try changing the threshold to:

- 7
- 9
- 10

Observe how the graph behavior changes.

This helps participants understand that workflow logic is under our control.


# 19. Enterprise Connection

This pattern is common in enterprise systems.

## DevOps

```text
Create RCA Draft
  ↓
Review RCA
  ↓
Quality Score >= 8?
  ├── Yes → Send to Manager
  └── No → Revise RCA
```

## Banking

```text
Customer Response Draft
  ↓
Compliance Review
  ↓
Compliant?
  ├── Yes → Send Response
  └── No → Rewrite Response
```

## Academic

```text
Question Paper Draft
  ↓
Moderator Review
  ↓
Approved?
  ├── Yes → Finalize
  └── No → Revise
```


# 20. Classroom Exercise

Create a LangGraph workflow for:

> Assignment Quality Review

State should include:

- topic
- assignment
- rubric
- score
- feedback
- revision_count

Flow:

```text
create_assignment
  ↓
review_assignment
  ↓
score >= 8?
   ├── yes → END
   └── no → revise_assignment → review_assignment
```


In [ ]:
# Exercise Starter Code

class AssignmentReviewState(TypedDict):
    topic: str
    assignment: str
    rubric: str
    score: int
    feedback: str
    revision_count: int

# TODO:
# 1. Create create_assignment node
# 2. Create review_assignment node
# 3. Create revise_assignment node
# 4. Create decision function
# 5. Add conditional edges
# 6. Compile and run graph


# 21. Key Takeaways

In this notebook, we learned:

- Conditional edges allow decision-based routing.
- A graph can loop back to earlier nodes.
- Loops are useful for revision and retries.
- State controls workflow behavior.
- Revision counters prevent infinite loops.
- LangGraph is useful when workflows are not simple straight lines.

---

## Final Mental Model

```text
State
  ↓
Node
  ↓
Decision
  ├── Path A
  └── Path B
        ↓
      Loop if needed
```

This completes our lightweight LangGraph section.

Next, we can move to MCP.
